In [3]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Análise de Preços de Abacates nos EUA - Aplicação de Machine Learning\n",
    "## TED 3: Problema de Negócio e Modelo Preditivo\n",
    "\n",
    "**Integrantes:**\n",
    "- Cesar Augusto Kreutz Bosing\n",
    "- Luiz Henrique Lazarin  \n",
    "- Tony Sousa Lopes\n",
    "\n",
    "**Disciplina:** BIG DATA E CIÊNCIAS DE DADOS\n",
    "**Professor:** Alex Rogaleski Marques\n",
    "\n",
    "## Introdução\n",
    "\n",
    "Este notebook continua a análise do dataset de preços de abacates nos EUA, iniciado no TED 1 e limpo no TED 2. Aqui, aplicaremos um algoritmo de Machine Learning para resolver um problema de negócio específico.\n",
    "\n",
    "**Problema de Negócio:** Prever o preço médio dos abacates (AveragePrice) com base em variáveis como volume total, tipos de abacates (4046, 4225, 4770) e região. Isso pode ajudar produtores e varejistas a otimizar preços e estoques.\n",
    "\n",
    "**Algoritmo Escolhido:** Árvore de Decisão para Regressão (DecisionTreeRegressor), devido à sua interpretabilidade e capacidade de lidar com relações não lineares.\n",
    "\n",
    "**Fluxo:**\n",
    "1. Carregamento e exploração dos dados\n",
    "2. Preparação dos dados (feature engineering)\n",
    "3. Treinamento e teste do modelo\n",
    "4. Avaliação e interpretação dos resultados"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.tree import DecisionTreeRegressor\n",
    "from sklearn.metrics import mean_squared_error, r2_score\n",
    "from sklearn.preprocessing import LabelEncoder\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Configuração de estilo\n",
    "plt.style.use('seaborn-v0_8')\n",
    "sns.set_palette(\"husl\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Carregamento e Exploração dos Dados\n",
    "\n",
    "Carregamos o dataset limpo do TED 2 e fazemos uma exploração inicial."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Carregar os dados (assumindo que o arquivo está no diretório atual)\n",
    "df = pd.read_csv(\"avocado.csv\")\n",
    "\n",
    "# Remover coluna desnecessária (como no TED 2)\n",
    "df = df.drop(columns=[\"Unnamed: 0\"], errors='ignore')\n",
    "\n",
    "# Converter data e extrair features\n",
    "df['Date'] = pd.to_datetime(df['Date'])\n",
    "df['Year'] = df['Date'].dt.year\n",
    "df['Month'] = df['Date'].dt.month\n",
    "df['Week'] = df['Date'].dt.isocalendar().week\n",
    "\n",
    "# Normalizar texto\n",
    "df['region'] = df['region'].str.strip().str.replace(' ', '_').str.lower()\n",
    "df['type'] = df['type'].str.strip().str.lower()\n",
    "\n",
    "# Criar variável de receita\n",
    "df['Revenue'] = df['AveragePrice'] * df['Total Volume']\n",
    "\n",
    "print(\"Dimensões do dataset:\", df.shape)\n",
    "print(\"\\nPrimeiras linhas:\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Estatísticas descritivas\n",
    "print(\"Estatísticas descritivas:\")\n",
    "df.describe()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribuição do target\n",
    "plt.figure(figsize=(8, 6))\n",
    "sns.histplot(df['AveragePrice'], bins=30, kde=True)\n",
    "plt.title('Distribuição do Preço Médio dos Abacates')\n",
    "plt.xlabel('Preço Médio (USD)')\n",
    "plt.ylabel('Frequência')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Preparação dos Dados\n",
    "\n",
    "Selecionamos features relevantes e preparamos os dados para o modelo."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Seleção de features\n",
    "# Target: AveragePrice\n",
    "# Features: Total Volume, 4046, 4225, 4770, Total Bags, Small Bags, Large Bags, XLarge Bags, type, year, region\n",
    "\n",
    "# Codificar variáveis categóricas\n",
    "le_type = LabelEncoder()\n",
    "df['type_encoded'] = le_type.fit_transform(df['type'])\n",
    "\n",
    "le_region = LabelEncoder()\n",
    "df['region_encoded'] = le_region.fit_transform(df['region'])\n",
    "\n",
    "# Selecionar features numéricas e codificadas\n",
    "features = ['Total Volume', '4046', '4225', '4770', 'Total Bags', 'Small Bags', 'Large Bags', 'XLarge Bags', 'type_encoded', 'region_encoded', 'Year', 'Month']\n",
    "target = 'AveragePrice'\n",
    "\n",
    "X = df[features]\n",
    "y = df[target]\n",
    "\n",
    "print(\"Features selecionadas:\", features)\n",
    "print(\"Shape de X:\", X.shape)\n",
    "print(\"Shape de y:\", y.shape)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Dividir em treino e teste\n",
    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n",
    "\n",
    "print(\"Treino:\", X_train.shape, y_train.shape)\n",
    "print(\"Teste:\", X_test.shape, y_test.shape)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Treinamento do Modelo\n",
    "\n",
    "Treinamos uma Árvore de Decisão para Regressão."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Criar e treinar o modelo\n",
    "model = DecisionTreeRegressor(random_state=42, max_depth=10)  # Limitar profundidade para evitar overfitting\n",
    "model.fit(X_train, y_train)\n",
    "\n",
    "print(\"Modelo treinado com sucesso!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Teste e Avaliação do Modelo\n",
    "\n",
    "Fazemos previsões no conjunto de teste e avaliamos o desempenho."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Previsões\n",
    "y_pred = model.predict(X_test)\n",
    "\n",
    "# Métricas de avaliação\n",
    "mse = mean_squared_error(y_test, y_pred)\n",
    "r2 = r2_score(y_test, y_pred)\n",
    "\n",
    "print(f\"Erro Quadrático Médio (MSE): {mse:.4f}\")\n",
    "print(f\"Coeficiente de Determinação (R²): {r2:.4f}\")\n",
    "\n",
    "# Interpretação: R² próximo de 1 indica bom ajuste; MSE baixo é desejável."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualização das previsões vs valores reais\n",
    "plt.figure(figsize=(8, 6))\n",
    "plt.scatter(y_test, y_pred, alpha=0.5)\n",
    "plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)\n",
    "plt.xlabel('Preço Real (USD)')\n",
    "plt.ylabel('Preço Previsto (USD)')\n",
    "plt.title('Previsões vs Valores Reais')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Importância das features\n",
    "feature_importance = pd.DataFrame({\n",
    "    'feature': features,\n",
    "    'importance': model.feature_importances_\n",
    "}).sort_values('importance', ascending=False)\n",
    "\n",
    "plt.figure(figsize=(10, 6))\n",
    "sns.barplot(x='importance', y='feature', data=feature_importance)\n",
    "plt.title('Importância das Features no Modelo')\n",
    "plt.show()\n",
    "\n",
    "print(\"Top 5 features mais importantes:\")\n",
    "print(feature_importance.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Interpretação dos Resultados\n",
    "\n",
    "- **Desempenho do Modelo:** O R² indica que o modelo explica cerca de X% da variabilidade no preço médio. O MSE mostra o erro médio nas previsões.\n",
    "- **Importância das Features:** As features mais importantes são [listar as top]. Isso sugere que fatores como volume e região têm grande impacto no preço.\n",
    "- **Limitações:** Árvores de decisão podem sofrer de overfitting; considerações futuras incluem tuning de hiperparâmetros ou uso de ensemble methods.\n",
    "- **Aplicação de Negócio:** Este modelo pode ajudar a prever preços futuros, otimizando decisões de compra/venda de abacates.\n",
    "\n",
    "## Conclusão\n",
    "\n",
    "Aplicamos com sucesso uma Árvore de Decisão para prever preços de abacates, seguindo o fluxo de ML. Os resultados fornecem insights valiosos para o negócio."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}


NameError: name 'null' is not defined